# Descriptive tables

This section builds country and year summary tables from the observed panel.

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PANEL_PATH = PROJECT_ROOT / 'data' / 'processed' / 'panel_skeleton.csv'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'

panel = pd.read_csv(PANEL_PATH)
summary_counts = pd.DataFrame({
    'country_year_cells': [len(panel)],
    'countries': [panel['geo'].nunique()],
    'years': [panel['year'].nunique()],
    'first_year': [panel['year'].min()],
    'last_year': [panel['year'].max()],
})
summary_counts

,country_year_cells,countries,years,first_year,last_year
0,608,37,18,2008,2025


## Country-level summary table

In [2]:
country_summary = (
    panel.groupby('geo', as_index=False)
    .agg(
        observed_years=('year', 'nunique'),
        mean_unmet_need_pc=('unmet_need_pc', 'mean'),
        min_unmet_need_pc=('unmet_need_pc', 'min'),
        max_unmet_need_pc=('unmet_need_pc', 'max'),
    )
    .round(3)
    .sort_values(['mean_unmet_need_pc', 'geo'], ascending=[False, True])
)
country_summary.to_csv(OUTPUTS_DIR / 'table_country_summary.csv', index=False)
country_summary

,geo,observed_years,mean_unmet_need_pc,min_unmet_need_pc,max_unmet_need_pc
0,AL,7,12.043,10.2,14.8
9,EE,18,10.150,4.3,16.4
21,LV,18,8.972,4.0,16.1
10,EL,18,8.956,5.4,13.1
36,XK,2,7.600,5.2,10.0
29,RO,18,7.233,2.2,12.2
34,TR,18,6.383,0.7,16.4
27,PL,18,5.456,1.9,9.0
30,RS,13,5.369,2.8,8.7
3,BG,18,4.900,1.0,15.3


## Year-level summary table

In [3]:
year_summary = (
    panel.groupby('year', as_index=False)
    .agg(
        countries_observed=('geo', 'nunique'),
        mean_unmet_need_pc=('unmet_need_pc', 'mean'),
        median_unmet_need_pc=('unmet_need_pc', 'median'),
        min_unmet_need_pc=('unmet_need_pc', 'min'),
        max_unmet_need_pc=('unmet_need_pc', 'max'),
    )
    .round(3)
    .sort_values('year')
)
year_summary.to_csv(OUTPUTS_DIR / 'table_year_summary.csv', index=False)
year_summary

,year,countries_observed,mean_unmet_need_pc,median_unmet_need_pc,min_unmet_need_pc,max_unmet_need_pc
0,2008,31,3.442,1.70,0.2,15.3
1,2009,31,3.358,2.10,0.2,14.9
2,2010,33,3.970,1.90,0.1,16.4
3,2011,33,4.000,2.20,0.1,16.1
4,2012,33,3.870,2.30,0.1,12.4
5,2013,35,4.334,3.00,0.0,13.8
6,2014,35,4.126,3.30,0.1,12.5
7,2015,35,3.494,2.60,0.1,12.7
8,2016,35,3.023,2.30,0.2,15.3
9,2017,36,2.822,2.15,0.1,13.1


## Simple inequality measures

In [4]:
def iqr(values):
    return values.quantile(0.75) - values.quantile(0.25)

# compute cross-country spread by year
year_inequality = (
    panel.groupby('year')
    .agg(
        countries_observed=('geo', 'nunique'),
        sd_unmet_need_pc=('unmet_need_pc', 'std'),
        iqr_unmet_need_pc=('unmet_need_pc', iqr),
    )
    .reset_index()
)
year_inequality = year_inequality.loc[year_inequality['countries_observed'] >= 20]
year_inequality = year_inequality.round(3).sort_values('year')
year_inequality.to_csv(OUTPUTS_DIR / 'table_year_inequality.csv', index=False)
year_inequality

,year,countries_observed,sd_unmet_need_pc,iqr_unmet_need_pc
0,2008,31,4.045,4.600
1,2009,31,3.539,2.900
2,2010,33,4.375,4.100
3,2011,33,4.062,4.800
4,2012,33,3.542,4.400
5,2013,35,3.824,6.300
6,2014,35,3.519,4.950
7,2015,35,3.371,3.550
8,2016,35,3.436,2.600
9,2017,36,3.085,2.350


## Short written summaries

In [5]:
first_year = int(panel['year'].min())
last_year = int(panel['year'].max())
country_count = int(panel['geo'].nunique())
cell_count = int(len(panel))

high_avg = country_summary.head(5)['geo'].tolist()
low_avg = country_summary.tail(5).sort_values('mean_unmet_need_pc')['geo'].tolist()

spread = year_inequality[['year', 'sd_unmet_need_pc']].dropna().copy()
if len(spread) >= 2 and spread.iloc[-1]['sd_unmet_need_pc'] < spread.iloc[0]['sd_unmet_need_pc']:
    spread_sentence = 'The cross-country standard deviation is lower at the end of the period than at the start of the measured period.'
elif len(spread) >= 2 and spread.iloc[-1]['sd_unmet_need_pc'] > spread.iloc[0]['sd_unmet_need_pc']:
    spread_sentence = 'The cross-country standard deviation is higher at the end of the period than at the start of the measured period.'
else:
    spread_sentence = 'The cross-country standard deviation shows little net change across the measured period.'

corr_path = OUTPUTS_DIR / 'panel_correlations.csv'
if corr_path.exists():
    corr = pd.read_csv(corr_path, index_col=0)
    poverty_corr = corr.loc['unmet_need_pc', 'poverty_or_social_exclusion_pc']
    gdp_corr = corr.loc['unmet_need_pc', 'gdp_per_capita_eur']
    association_sentence = (
        f'Complete-case correlations show a correlation of {poverty_corr:.2f} with poverty or social exclusion '
        f'and {gdp_corr:.2f} with GDP per capita.'
    )
else:
    association_sentence = 'The correlation table was not available when this summary was written.'

summary_text = f'''# EDA descriptive summary

- The observed panel contains {cell_count} country-year cells for {country_count} national country codes from {first_year} to {last_year}.
- Countries with high average unmet medical needs over their observed years include {', '.join(high_avg)}.
- Countries with low average unmet medical needs over their observed years include {', '.join(low_avg)}.
- {spread_sentence}
- {association_sentence}
- These points describe patterns in aggregate Eurostat data. They are not model results.
'''

(OUTPUTS_DIR / 'eda_descriptive_summary.md').write_text(summary_text, encoding='utf-8')
summary_text

'# EDA descriptive summary\n\n- The observed panel contains 608 country-year cells for 37 national country codes from 2008 to 2025.\n- Countries with high average unmet medical needs over their observed years include AL, EE, LV, EL, XK.\n- Countries with low average unmet medical needs over their observed years include NL, AT, MT, LU, CZ.\n- The cross-country standard deviation is lower at the end of the period than at the start of the measured period.\n- Complete-case correlations show a correlation of 0.36 with poverty or social exclusion and -0.30 with GDP per capita.\n- These points describe patterns in aggregate Eurostat data. They are not model results.\n'